## 1. Install Pydantic AI

In [1]:
!pip install pydantic-ai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.2/101.2 kB 5.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-instrumentation to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opentelemetry-instrumentation-httpx to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-instrumentation-httpx to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/b

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

## 2. Basic Agent

In [ ]:
from pydantic_ai import Agent

agent = Agent(
    "openai:gpt-4o-mini",
    instructions="Answer in 1-2 sentences."
)

result = agent.run_sync("What is a large language model?")
print(result.output)

## 3. Structured Output


`from pydantic_ai import Agent` is repeated across cells for clarity.

In [ ]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent

class JobPosting(BaseModel):
    job_title: str
    company_name: str
    required_skills: list[str] = Field(description="Technical skills")
    seniority_level: str
    is_remote: bool

agent = Agent(
    "openai:gpt-4o-mini",
    output_type=JobPosting,
    instructions="Extract structured job info."
)

text = """
We are hiring a Senior Python Engineer at CoolData Inc.
The role is fully remote. Required: FastAPI, PostgreSQL, Docker.
"""

result = agent.run_sync(text)
print(result.output)

## 4. Tools

In [ ]:
import json
from pydantic import BaseModel
from pydantic_ai import Agent

NUTRITION_DB = {
    "chicken breast": {"calories": 165, "protein_g": 31, "carbs_g": 0, "fat_g": 3.6},
    "brown rice": {"calories": 216, "protein_g": 5, "carbs_g": 45, "fat_g": 1.8},
    "broccoli": {"calories": 55, "protein_g": 3.7, "carbs_g": 11, "fat_g": 0.6},
    "olive oil": {"calories": 119, "protein_g": 0, "carbs_g": 0, "fat_g": 13.5},
}

class MealSummary(BaseModel):
    total_calories: int
    total_protein_g: float
    total_carbs_g: float
    total_fat_g: float
    health_verdict: str
    recommendation: str

agent = Agent(
    "openai:gpt-4o-mini",
    output_type=MealSummary,
    instructions="Use tools to compute totals."
)

@agent.tool_plain
def get_ingredient_nutrition(ingredient: str) -> str:
    """Return nutrition per 100g."""
    data = NUTRITION_DB.get(ingredient.lower().strip())
    if data:
        return json.dumps({"ingredient": ingredient, **data})
    return "Not found"

result = agent.run_sync(
    "Analyse: 200g chicken breast, 150g brown rice, 100g broccoli"
)

print(result.output)

## 5. Dependency Injection

In [ ]:
from dataclasses import dataclass
from pydantic_ai import Agent, RunContext

@dataclass
class NutritionService:
    database: dict

    def lookup(self, ingredient: str):
        return self.database.get(ingredient.lower().strip())

agent = Agent(
    "openai:gpt-4o-mini",
    output_type=MealSummary,
    deps_type=NutritionService,
    instructions="Use tools with deps."
)

@agent.tool
def get_nutrition(ctx: RunContext[NutritionService], ingredient: str) -> str:
    """Lookup nutrition."""
    data = ctx.deps.lookup(ingredient)
    return json.dumps(data) if data else "Not found"

service = NutritionService(database=NUTRITION_DB)

result = agent.run_sync(
    "Analyse: 150g chicken breast",
    deps=service
)

print(result.output)

## 6. Capabilities (Web + Thinking)

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.capabilities import WebSearch, Thinking

agent = Agent(
    "openai:gpt-4o-mini",
    capabilities=[WebSearch(), Thinking(effort="high")]
)

result = agent.run_sync("Latest AI research trends")
print(result.output)